Evaluate inference of fresh subjects who don't have ground truth segmentations

In [15]:
%matplotlib widget
import matplotlib.pyplot as plt

from scripts import lesion_diagnostics
import os
from pathlib import Path
import pandas as pd
import numpy as np
import nibabel as nib
import json
import pickle as pkl
from collections import defaultdict

import scripts.compile_run_metrics
import helpers.utils
import scripts.inference
from IPython.display import clear_output

from reload_recursive import reload_recursive
from tqdm.notebook import tqdm


In [16]:
reload_recursive(scripts.compile_run_metrics)
#TODO Thi
from scripts.compile_run_metrics import EXPERIMENT_KEYS

reload_recursive(scripts.inference)
from scripts.inference import uncrop_predictions

reload_recursive(helpers.utils)
from helpers.utils import dice_score, get_prl_indices

import helpers.shell_interface
reload_recursive(helpers.shell_interface)
from helpers.shell_interface import open_itksnap_workspace_cmd

import core.dataset
reload_recursive(core.dataset)
from core.dataset import Dataset

import core.experiment
reload_recursive(core.experiment)
from core.experiment import Experiment
from loguru import logger

import scripts
reload_recursive(scripts)
from scripts import lesion_diagnostics as lesion_dx
from scripts.compile_run_metrics import load_or_cache_run

In [17]:
def get_infer_path(dataset, test_case, experiment_name) -> Path:
    case_dir = dataset.lesion_dir(test_case)
    matches = list(case_dir.glob(f"*{experiment_name.replace('/','_')}.nii.gz"))
    if len(matches) > 1:
        logger.warning(f"Found more than 1 case: {','.join(matches)}, returning the first")
    return matches[0]
    
def find_inference(search_path, experiment_name) -> Path:
    matches = list(search_path.glob(f"*{experiment_name.replace('/','_')}.nii.gz"))
    if len(matches) > 1:
        logger.warning(f"Found more than 1 case: {','.join(matches)}, returning the first")
    return matches[0]



In [23]:
import math
import traceback

def analyze_prl_case(prl_case: dict, subject_dir, bbox_suffix):
    subid = prl_case['subid']
    index = prl_case['lesion_index']
    data_root = subject_dir.parent

    lesion_index_path = subject_dir / "lstai_lesion_index.nii.gz"
    lesion_index = nib.load(str(lesion_index_path)).get_fdata().astype(np.int32)

    # Parse bounding boxes
    bbox_file = subject_dir / f"lstai_bounding_boxes{bbox_suffix}.txt"
    bounding_boxes = lesion_diagnostics._parse_bounding_boxes(bbox_file)

    try:
        assert bounding_boxes[index-1][0] == index
    except AssertionError:
        print(index, bounding_boxes[index-1][0])
    coords = bounding_boxes[index-1][1]

    # Load inference output for this ROI
    infer_path = data_root / prl_case["inference"]
    if not infer_path.exists():
        print(f"Inference output not found: {infer_path}")
        return None
    label_paths = [infer_path]
    
    label_keys = ["infer"]
    lesion_stats = {"lesion_index": index, **{f"rim_voxels_{k}": None for k in label_keys}}
    lesion_data = {"subid": subid, "lesion_index": index}
    for id, lab in zip(label_keys, label_paths):
        lab_nifti = nib.load(str(lab))
        lab_data = lab_nifti.get_fdata().astype(np.uint8)
        voxel_size = lab_nifti.header.get_zooms()[:3]
        voxel_volume = math.prod(voxel_size)

        lesion_data[f"label_{id}"] = lab_data

        # Crop lesion index with the same bounding box
        index_crop = lesion_diagnostics._crop_from_volume(lesion_index, coords)
        lesion_data[f"index_crop_{id}"] = index_crop

        try:
            # any iron detection: rim voxels that overlap the central lesion's footprint
            has_iron = np.any((index_crop == index) & (lab_data == 2))
            if id == "infer":
                lesion_stats['has_iron_infer'] = has_iron
            # ---Process Rim---
            # get the rim for the center lesion
            rim = lesion_dx._get_lesion_rim(index_crop, lab_data, index)
            rim_count = int(rim.sum())
            rim_sphere_radius = lesion_dx.rim_enclosing_sphere_radius(rim, voxel_size)

            # get convex hull
            hull = lesion_dx.get_convex_hull(rim, voxel_sizes=voxel_size)
            
            lesion_data.update({
                f"rim_{id}": rim,
                f"rim_hull_{id}": hull,
            })

            lesion_stats.update({
                f"rim_voxels_{id}": rim_count,
                f"rim_volume_{id}": rim_count*voxel_volume,
            })

            lesion_stats.update({
                f"rim_hull_volume_{id}": hull.volume,
                f"rim_sphere_radius_{id}": rim_sphere_radius,
            })
        except Exception:
            print(prl_case['subid'], prl_case['lesion_index'])
            tb_str = traceback.format_exc()
            print(f"Captured Traceback:\n{tb_str}")
            pass

        # ---Process T2 Lesion---
        try:
            lesion = lesion_dx._get_center_lesion(index_crop, lab_data, index)
            lesion_count = int(lesion.sum())

            # get convex hull
            hull = lesion_dx.get_convex_hull(lesion, voxel_sizes=voxel_size)
            
            lesion_data.update({
                f"lesion_{id}": lesion,
                f"lesion_hull_{id}": hull,
            })

            lesion_stats.update({
                f"lesion_voxels_{id}": lesion_count,
                f"lesion_volume_{id}": lesion_count*voxel_volume,
            })
            lesion_stats.update({
                f"lesion_hull_volume_{id}": hull.volume
            })
        except Exception:
            tb_str = traceback.format_exc()
            print(f"Captured Traceback:\n{tb_str}")
            pass

        lesion_data[f"voxel_size_{id}"] = voxel_size    
    return lesion_stats, lesion_data

Define which experiment's model to use for inference

In [19]:
dataset_name = "roi_train2"
dataset = Dataset(dataset_name)

experiment_key, run_name = "stage3", "run2"
experiment_name = f"{EXPERIMENT_KEYS[experiment_key]}/{run_name}"
experiment = Experiment.from_run_dir(experiment_name, dataset)

expand_xy: int = experiment.preprocess_config.expand_xy
expand_z: int = experiment.preprocess_config.expand_z
images: tuple[str, ...] = experiment.preprocess_config.images

In [20]:

inference_dataset = Dataset("inference_dataset")
prl_df = inference_dataset.prl_df
datalist_df = pd.DataFrame(inference_dataset.datalist_template['testing']).set_index(["subid", "lesion_index"])

subid = 1082
lesion_index = 10
iloc = datalist_df.index.get_loc((subid, lesion_index))
case_dict = datalist_df.reset_index().iloc[iloc].to_dict()

inference_dataset.datalist_template['testing']
test_case = inference_dataset.datalist_template['testing'][0]
inference = get_infer_path(inference_dataset, test_case, experiment_name)
images = inference_dataset.get_images(test_case, ["flair", "phase"])
case_dict["inference"] = inference

lst_lesion_index = inference_dataset.subject_dir(subid) / "lstai_lesion_index.nii.gz"
labels = [inference, lst_lesion_index]

In [21]:
subid = 1082
subject_prls = get_prl_indices(prl_df, subid)
for index in subject_prls:
    search_path = inference_dataset.subject_dir(subid) / str(index)
    inference = find_inference(search_path, experiment_name)

In [25]:
len(inference_dataset.datalist_template['testing'])

3912

In [28]:
suffix = "_" + experiment.preprocess_config.suffix.strip("_")

full_data = defaultdict(list)
for test_case in tqdm(inference_dataset.datalist_template['testing']):
    subid = test_case['subid']
    index = test_case['lesion_index']
    subject_dir = inference_dataset.subject_dir(subid)
    search_path = subject_dir / str(index)
    inference = find_inference(search_path, experiment_name)
    if not inference.exists():
        print(f"{inference} doesn't exist")
        continue
    test_case["inference"] = inference
    full_data['case_info'].append(test_case)
    prl_stats, prl_data = analyze_prl_case(test_case, subject_dir, suffix)
    full_data['prl_stats'].append(prl_stats)
    full_data['prl_data'].append(prl_data)
    

  0%|          | 0/3912 [00:00<?, ?it/s]

1156 12
Captured Traceback:
Traceback (most recent call last):
  File "/tmp/ipykernel_1882847/1589030955.py", line 69, in analyze_prl_case
    f"rim_hull_volume_{id}": hull.volume,
                             ^^^^^^^^^^^
AttributeError: 'NoneType' object has no attribute 'volume'

1296 50
Captured Traceback:
Traceback (most recent call last):
  File "/tmp/ipykernel_1882847/1589030955.py", line 69, in analyze_prl_case
    f"rim_hull_volume_{id}": hull.volume,
                             ^^^^^^^^^^^
AttributeError: 'NoneType' object has no attribute 'volume'

Captured Traceback:
Traceback (most recent call last):
  File "/tmp/ipykernel_1882847/1589030955.py", line 96, in analyze_prl_case
    f"lesion_hull_volume_{id}": hull.volume
                                ^^^^^^^^^^^
AttributeError: 'NoneType' object has no attribute 'volume'

1183 19
Captured Traceback:
Traceback (most recent call last):
  File "/tmp/ipykernel_1882847/1589030955.py", line 69, in analyze_prl_case
    f"rim_hull_

In [29]:
analysis_dir = Path("/home/srs-9/Projects/prl_project/analysis/")
cache_dir = analysis_dir / ".cache"
full_data_path = cache_dir / f"prl_inference_image_stats-{experiment.id.replace('/', '_')}.pkl"
stats_data_path = analysis_dir / f"prl_inference_image_stats-{experiment.id.replace('/', '_')}.csv"

with open(full_data_path, 'wb') as f:
    pkl.dump(full_data, f)

df_data = []
for case_info, case_stats in zip(full_data['case_info'], full_data['prl_stats']):
    df_data.append({**case_info, **case_stats})

df_cases = pd.DataFrame(df_data).set_index(["subid","lesion_index"])
df_cases.sort_values(by="case_type", ascending=False).to_csv(stats_data_path)

In [31]:
df_cases.index[0]

(np.int64(1126), np.int64(4))

In [52]:
import numpy as np
from sklearn.decomposition import PCA
from scipy import ndimage

def compute_pca_features(rim_mask):
    coords = np.argwhere(rim_mask > 0)
    if len(coords) < 3:
        return {
            "pca_size": np.nan,
            "pca_sphericity": np.nan,
            "pca_planarity": np.nan,
            "pca_linearity": np.nan,
        }

    pca = PCA(n_components=3)
    pca.fit(coords)

    comps = sorted(pca.singular_values_, reverse=True)

    features = {}
    features["pca_size"] = np.sum(comps)
    features["pca_sphericity"] = comps[2] / comps[0] if comps[0] > 0 else np.nan
    features["pca_planarity"] = (comps[1] - comps[2]) / comps[0] if comps[0] > 0 else np.nan
    features["pca_linearity"] = (comps[0] - comps[1]) / comps[0] if comps[0] > 0 else np.nan
    return features


def compute_radial_metrics(rim_mask):
    coords = np.argwhere(rim_mask > 0)

    if len(coords) == 0:
        return {
            "mean_radius": np.nan,
            "std_radius": np.nan,
            "min_radius": np.nan,
            "max_radius": np.nan,
            "radial_cv": np.nan,
        }

    centroid = ndimage.center_of_mass(rim_mask)
    dists = np.linalg.norm(coords - centroid, axis=1)

    features = {}
    features["mean_radius"] = dists.mean()
    features["std_radius"] = dists.std()
    features["min_radius"] = dists.min()
    features["max_radius"] = dists.max()
    features["radial_cv"] = (
        features["std_radius"] / features["mean_radius"]
        if features["mean_radius"] > 0 else np.nan
    )
    return features

In [50]:
reload_recursive(helpers.utils)
from helpers.utils import get_prl_indices

prl_def_prob = []
prl_all = []
for subid in inference_dataset.subjects:
    prl_inds = get_prl_indices(prl_df, subid, possible=True)
    prl_all.append(prl_inds)
    for ind in prl_inds:
        if ind <= 0:
            continue
        if df_cases.loc[(subid, ind), "case_type"] != "PRL":
            print(subid, ind)
            df_cases.loc[(subid, ind), "case_type"] = "PRL"
            
df_cases.to_csv(analysis_dir / f"prl_inference_defprobposs_image_stats-{experiment.id.replace('/', '_')}.csv")

In [57]:
df_data = []
for case_info, case_stats in zip(full_data['case_info'], full_data['prl_stats']):
    df_data.append({**case_info, **case_stats})

df_cases = pd.DataFrame(df_data).set_index(["subid","lesion_index"])
prl_def_prob = []
prl_all = []
for subid in inference_dataset.subjects:
    prl_inds = get_prl_indices(prl_df, subid, possible=True)
    prl_all.append(prl_inds)
    for ind in prl_inds:
        if ind <= 0:
            continue
        if df_cases.loc[(subid, ind), "case_type"] != "PRL":
            print(subid, ind)
            df_cases.loc[(subid, ind), "case_type"] = "PRL"
iron_cases = df_cases[df_cases['has_iron_infer']]

df_cases2 = df_cases.copy()
i = 0
j = 0
for subid, lesion_index in iron_cases.index:
    iloc = df_cases.index.get_loc((subid, lesion_index))
    prl_data = full_data['prl_data'][iloc]
    try:
        assert prl_data['subid'] == subid
        assert prl_data['lesion_index'] == lesion_index
    except AssertionError:
        print(f"assertion error {iloc}")
    rim_mask = prl_data['rim_infer']
    try:
        features = compute_pca_features(rim_mask)
    except Exception:
        print(subid, lesion_index)
        tb_str = traceback.format_exc()
        print(f"Captured Traceback:\n{tb_str}")
        features = {}
    else:
        i += 1
    try:
        features = {
            **features,
            **compute_radial_metrics(rim_mask),
        }
    except Exception:
        print(subid, lesion_index)
        tb_str = traceback.format_exc()
        print(f"Captured Traceback:\n{tb_str}")
    else:
        j += 1
    df_cases2.loc[(subid,lesion_index), list(features.keys())] = list(features.values())
print(i,j)

1082 6
1118 2
1118 4
1126 9
1126 15
1133 1
1164 4
1183 17
1183 25
1186 5
1234 2
1234 4
1239 2
1252 63
1257 19
1265 36
1265 57
1265 123
1281 120
1282 5
1282 6
1292 5
1296 3
1296 14
1316 6
1316 1
1327 8
1327 2
1353 2
1353 4
1353 5
1523 34
1537 7
2017 8
2087 2
2098 9
2122 13
2168 2
2168 10
2207 1
520 520


In [58]:
df_cases2.to_csv(analysis_dir / f"prl_inference_defprobposs_image_stats-{experiment.id.replace('/', '_')}.csv")

In [24]:
subid = 1082
index = 10
iloc = datalist_df.index.get_loc((subid, index))
case_dict = datalist_df.reset_index().iloc[iloc].to_dict()
subject_dir = inference_dataset.subject_dir(subid)
search_path = subject_dir / str(index)
inference = find_inference(search_path, experiment_name)

case_dict["inference"] = inference
suffix = "_" + experiment.preprocess_config.suffix.strip("_")
prl_data = analyze_prl_case(case_dict, subject_dir, suffix)

images = inference_dataset.get_images(case_dict, ["flair", "phase"], suffix)
labels = [inference]

cmd = open_itksnap_workspace_cmd(images, labels, rename_root=("/media/smbshare", "Z:/"))
print(cmd)

itksnap -g Z:/srs-9/prl_project/inference_data/sub1082-20161218/10/flair_xy20_z2.nii.gz -o Z:/srs-9/prl_project/inference_data/sub1082-20161218/10/phase_xy20_z2.nii.gz -s Z:/srs-9/prl_project/inference_data/sub1082-20161218/10/flair.phase_xy20_z2_infer_stage3_numcrops_bkd_constwt115_run2.nii.gz


In [ ]:
import pyperclip

cmd = open_itksnap_workspace_cmd(images, labels, rename_root=("/media/smbshare", "Z:/"))
pyperclip.copy(cmd)

In [ ]:
list(case_dir.glob(f"*{experiment_name.replace("/","_")}.nii.gz"))

[PosixPath('/media/smbshare/srs-9/prl_project/inference_data/sub1126-20170609/4/flair.phase_xy20_z2_infer_stage3_numcrops_bkd_constwt115_run2.nii.gz')]